In [18]:
import pandas as pd
import os
import matplotlib.pyplot as plt
import numpy as np
import folium
from math import radians, sin, cos, sqrt, atan2
from matplotlib import colormaps
from matplotlib.colors import to_hex
import haversine

## Take-aways for our actual application

Keep in mind:
- some trip times include the time the taxi has been on stand by and then become much longer -> we need to make sure to filter out those parts
- also keep i nmind that locations are only send every 10 seconds -> quite large jumps on the map

Maybe add these features to our dashboard:
- Identify when a taxi starts a new trip
- avergae trip duration in dashboard
- average trip length in dashboard

## Analysis

In [19]:
# read all txt files and put them into dataframe
data_dir = "/home/geschen/Documents/big_data/bd26_project_w4_b/provider/data/release/taxi_log_2008_by_id"
txt_files = sorted(
        os.path.join(data_dir, f)
        for f in os.listdir(data_dir)
        if f.endswith(".txt")
    )

In [20]:

df = pd.DataFrame()
for file in txt_files[:2]:
    with open(file, encoding="utf-8", errors="replace") as fh:
        for line in fh:
            temp = {}
            line = line.strip()
            if not line:
                continue
            parts = line.split(",")
            if len(parts) != 4:
                print("Skipping malformed line in %s: %r", path, line)
                continue
            taxi_id_str, ts_str, lon_str, lat_str = parts
            temp["taxi_id_str"] = [taxi_id_str]
            temp["ts_str"] = [ts_str]
            temp["lon_str"] = [lon_str]
            temp["lat_str"] = [lat_str]
            temp = pd.DataFrame.from_dict(temp)
            df = pd.concat([temp, df])

Cannot put all taxis into one single dataframe
But need to process them one by one

In [21]:
df

,taxi_id_str,ts_str,lon_str,lat_str
0,10,2008-02-08 17:38:00,116.38552,39.90773
0,10,2008-02-08 17:37:55,116.38562,39.90737
0,10,2008-02-08 17:37:24,116.38548,39.90558
0,10,2008-02-08 17:36:01,116.3855,39.90552
0,10,2008-02-08 17:34:55,116.3855,39.89905
...,...,...,...,...
0,1,2008-02-02 16:06:08,116.47186,39.91248
0,1,2008-02-02 15:56:08,116.51627,39.91034
0,1,2008-02-02 15:46:08,116.51135,39.93883
0,1,2008-02-02 15:46:08,116.51135,39.93883


# Initial analysis

Look at:
- How long is the overall period
- look at the speed
- look at the average time a taxi is driving
- look at geographical outliers
- look at jumps in the locations

In [22]:
tx_1 = df[df["taxi_id_str"] == "1"]

In [23]:
tx_1

,taxi_id_str,ts_str,lon_str,lat_str
0,1,2008-02-08 15:51:31,116.54723,39.90841
0,1,2008-02-08 15:41:31,116.57156,39.90263
0,1,2008-02-08 15:31:31,116.53174,39.91536
0,1,2008-02-08 15:21:31,116.50789,39.93128
0,1,2008-02-08 15:11:31,116.48347,39.91954
...,...,...,...,...
0,1,2008-02-02 16:06:08,116.47186,39.91248
0,1,2008-02-02 15:56:08,116.51627,39.91034
0,1,2008-02-02 15:46:08,116.51135,39.93883
0,1,2008-02-02 15:46:08,116.51135,39.93883


In [24]:
tx_1["lat_str"].info(
)

<class 'pandas.Series'>
Index: 588 entries, 0 to 0
Series name: lat_str
Non-Null Count  Dtype
--------------  -----
588 non-null    str  
dtypes: str(1)
memory usage: 9.2 KB


In [25]:
tx_1["ts_str"] = pd.to_datetime(tx_1["ts_str"], format="%Y-%m-%d %H:%M:%S")
tx_1["unix_time"] = tx_1["ts_str"].astype("int64") // 10**6

In [45]:
df["ts_str"] = pd.to_datetime(df["ts_str"], format="%Y-%m-%d %H:%M:%S")
df["unix_time"] = df["ts_str"].astype("int64") // 10**6

In [47]:
def haversine(lat1, lon1, lat2, lon2):
    """Returns distance in kilometers between two GPS points."""
    R = 6371  # Earth radius in km
    lat1, lon1, lat2, lon2 = map(radians, [lat1, lon1, lat2, lon2])
    dlat = lat2 - lat1
    dlon = lon2 - lon1
    a = sin(dlat/2)**2 + cos(lat1) * cos(lat2) * sin(dlon/2)**2
    return R * 2 * atan2(sqrt(a), sqrt(1 - a))

def get_average_trip_duration(df, gap_threshold_seconds=300):
    df = df.sort_values("unix_time").reset_index(drop=True)
    df["lon"] = df["lon_str"].astype(float)
    df["lat"] = df["lat_str"].astype(float)

    trips = []
    trip_start     = df.loc[0, "unix_time"]
    prev_time      = df.loc[0, "unix_time"]
    prev_lat       = df.loc[0, "lat"]
    prev_lon       = df.loc[0, "lon"]
    trip_distance  = 0.0
    trip_coords    = [(df.loc[0, "lat"], df.loc[0, "lon"])]  # collect coords per trip

    for i in range(1, len(df)):
        curr_time = df.loc[i, "unix_time"]
        curr_lat  = df.loc[i, "lat"]
        curr_lon  = df.loc[i, "lon"]
        diff_seconds = curr_time - prev_time

        trip_distance += haversine(prev_lat, prev_lon, curr_lat, curr_lon)
        trip_coords.append((curr_lat, curr_lon))

        if diff_seconds > gap_threshold_seconds:
            trip_duration = prev_time - trip_start
            trips.append({
                "trip_start":       trip_start,
                "trip_end":         prev_time,
                "duration_seconds": trip_duration,
                "duration_minutes": trip_duration / 60,
                "distance_km":      trip_distance,
                "coords":           trip_coords        # store coords
            })
            trip_start    = curr_time
            trip_distance = 0.0
            trip_coords   = [(curr_lat, curr_lon)]     # reset with first point of new trip

        prev_time = curr_time
        prev_lat  = curr_lat
        prev_lon  = curr_lon

    # Last trip
    trip_duration = prev_time - trip_start
    trips.append({
        "trip_start":       trip_start,
        "trip_end":         prev_time,
        "duration_seconds": trip_duration,
        "duration_minutes": trip_duration / 60,
        "distance_km":      trip_distance,
        "coords":           trip_coords
    })

    trips_df = pd.DataFrame(trips)
    # Keep all trips including distance ~0 (breaks), only drop duration == 0
    trips_df = trips_df[trips_df["duration_seconds"] > 0]

    avg_duration_minutes = trips_df["duration_minutes"].mean()
    avg_distance_km      = trips_df["distance_km"].mean()

    print(f"Number of trips found:  {len(trips_df)}")
    print(f"Average trip duration:  {avg_duration_minutes:.1f} minutes")
    print(f"Average trip distance:  {avg_distance_km:.2f} km")

    return avg_duration_minutes, avg_distance_km, trips_df

In [48]:
def get_trips_all_taxis(df, gap_threshold_seconds=300):
    all_trips  = []
    summary    = []

    for taxi_id, taxi_df in df.groupby("taxi_id_str"):
        taxi_df = taxi_df.reset_index(drop=True)
        print(f"\n--- Taxi {taxi_id} ---")
        avg_duration, avg_distance, trips_df = get_average_trip_duration(
            taxi_df, gap_threshold_seconds=gap_threshold_seconds
        )
        trips_df["taxi_id_str"] = taxi_id  # tag each trip with its taxi
        all_trips.append(trips_df)
        summary.append({
            "taxi_id":             taxi_id,
            "avg_duration_minutes": avg_duration,
            "avg_distance_km":      avg_distance,
        })

    all_trips_df = pd.concat(all_trips).reset_index(drop=True)
    summary_df   = pd.DataFrame(summary)

    return all_trips_df, summary_df

In [49]:
all_trips_df, summary_df = get_trips_all_taxis(df, gap_threshold_seconds=300)


--- Taxi 1 ---
Number of trips found:  2
Average trip duration:  4.1 minutes
Average trip distance:  3.22 km

--- Taxi 10 ---
Number of trips found:  60
Average trip duration:  45.2 minutes
Average trip distance:  19.67 km


In [50]:
all_trips_df

,trip_start,trip_end,duration_seconds,duration_minutes,distance_km,coords,taxi_id_str
0,1202209714,1202209956,242,4.033333,1.339784,"[(39.915, 116.46515), (39.91569, 116.45519), (...",1
1,1202382088,1202382335,247,4.116667,5.096160,"[(39.90824, 116.59778), (39.91294, 116.57738),...",1
2,1201959123,1201960373,1250,20.833333,5.209671,"[(39.92157, 116.44457), (39.9219, 116.44043), ...",10
3,1201967340,1201968065,725,12.083333,3.422507,"[(39.91153, 116.39868), (39.91143, 116.39842),...",10
4,1201968780,1201973381,4601,76.683333,33.508041,"[(39.89552, 116.40897), (39.89512, 116.40928),...",10
...,...,...,...,...,...,...,...
57,1202479341,1202479979,638,10.633333,6.149535,"[(39.9089, 116.35042), (39.90978, 116.35048), ...",10
58,1202483725,1202484848,1123,18.716667,2.845933,"[(39.95283, 116.36313), (39.95247, 116.36313),...",10
59,1202485587,1202485627,40,0.666667,0.060356,"[(39.94138, 116.35218), (39.94145, 116.35228),...",10
60,1202486272,1202488164,1892,31.533333,14.736353,"[(39.94158, 116.35273), (39.94158, 116.35273),...",10


In [31]:
def plot_trips_on_map(trips_df):
    trips_df = trips_df.reset_index(drop=True)  # fix index gaps
    
    all_coords = [coord for coords in trips_df["coords"] for coord in coords]
    map_centre = [
        np.mean([c[0] for c in all_coords]),
        np.mean([c[1] for c in all_coords])
    ]
    m = folium.Map(location=map_centre, zoom_start=13)

    durations      = trips_df["duration_minutes"].values
    norm_durations = (durations - durations.min()) / (durations.max() - durations.min() + 1e-9)
    cmap           = colormaps["RdYlBu_r"]

    for i, row in trips_df.iterrows():  # i now matches norm_durations index cleanly
        color        = to_hex(cmap(norm_durations[i]))
        duration_min = row["duration_minutes"]
        distance_km  = row["distance_km"]
        coords       = row["coords"]

        if distance_km < 1:
            folium.CircleMarker(
                location=coords[0],
                radius=6,
                color=color,
                fill=True,
                fill_opacity=0.8,
                tooltip=f"Break — {duration_min:.1f} min, {distance_km:.3f} km"
            ).add_to(m)
        else:
            folium.PolyLine(
                locations=coords,
                color=color,
                weight=4,
                opacity=0.8,
                tooltip=f"Trip {i + 1}: {duration_min:.1f} min, {distance_km:.2f} km"
            ).add_to(m)

    legend_html = """
    <div style="position: fixed; bottom: 40px; left: 40px; z-index: 1000;
                background-color: white; padding: 10px; border-radius: 8px;
                box-shadow: 2px 2px 6px rgba(0,0,0,0.3); font-size: 13px;">
        <b>Trip Duration</b><br>
        <span style="color:#4575b4;">&#9644;</span> Short<br>
        <span style="color:#fee090;">&#9644;</span> Medium<br>
        <span style="color:#d73027;">&#9644;</span> Long<br><br>
        <b>&#9711;</b> Stationary break
    </div>
    """
    m.get_root().html.add_child(folium.Element(legend_html))

    output_path = "trips_map.html"
    m.save(output_path)
    print(f"Map saved to {output_path}")
    return m

In [13]:
trips.index

Index([ 0,  1,  2,  3,  4,  5,  6,  7,  8,  9, 11, 12, 13, 14, 15, 16, 17, 18,
       19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36,
       37, 38, 40, 41, 43, 44],
      dtype='int64')

In [32]:
m = plot_trips_on_map(trips)

Map saved to trips_map.html
